# Практика 21 · Swin і гібриди

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ **Цей зошит навчає девʼять мереж.** Заміряно: **близько чотирьох з половиною
> хвилин** (265 секунд у контрольному прогоні) на чотирьох ядрах без відеокарти,
> в один потік. Перші сім замірів проходять за секунди — вони не потребують
> жодного навчання. Довга тільки остання клітинка з порівнянням трьох моделей:
> девʼять навчань, близько трьох з половиною хвилин.

Порядок такий: спершу все, що доводиться алгеброю, потім єдиний замір із навчанням.

**Без навчання:**

1. Ціна вікон проти повної уваги: чому квадратична залежність стає лінійною.
2. Власна реалізація віконної уваги — і перевірка, що при вікні на всю сітку
   вона збігається з `nn.MultiheadAttention`.
3. Доказ, що вікна не спілкуються: змінюємо один токен і рахуємо, скільки
   токенів на виході змінилось.
4. Зсунуті вікна з циклічним зсувом і маскою — і той самий підрахунок.
5. Ієрархія: обʼєднання патчів, форми наскрізь, порівняння з ResNet.
6. Розбір справжнього `swin_t` по стадіях і по типах шарів.
7. Розбір `convnext_tiny`: що в ній від згортки, а що від трансформера.

**З навчанням:**

8. Три моделі однакового розміру на нашій задачі: повна увага, вікна, згортка.


In [ ]:
import time
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# Один потік: на дрібних тензорах він і швидший за чотири, і — головне —
# детермінований. Під кількома потоками додавання float іде в іншому порядку.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch      :", torch.__version__)
print("numpy      :", np.__version__)
print("потоків CPU:", torch.get_num_threads())

## Дані: ті самі шість фігур

Датасет той самий, що в блоках 2 і 3 та в темі 20: шість класів фігур 28×28,
згенерованих формулами. Нічого не завантажується. Шум 0.45, центр гуляє
на ±5 пікселів.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]


def draw_shape(kind, rng, size=28, jitter=5, noise=0.45):
    '''Малює одну фігуру заданого класу як масив 28×28 зі значеннями 0..1.'''
    image = np.zeros((size, size), dtype=np.float32)
    # центр зсуваємо, щоб мережа не завчила одне-єдине положення предмета
    center_y = size / 2 + rng.integers(-jitter, jitter + 1)
    center_x = size / 2 + rng.integers(-jitter, jitter + 1)
    radius = rng.integers(5, 9)

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius) & (distance >= (radius - 3) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, noise, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, rng):
    '''Повертає (count, 1, 28, 28) і (count,). Класи чергуються, тож вони збалансовані.'''
    images = np.zeros((count, 1, 28, 28), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = i % 6
        images[i, 0] = draw_shape(kind, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


rng = np.random.default_rng(42)
train_x, train_y = make_dataset(1000, rng)
test_x, test_y = make_dataset(600, rng)

print("навчальна вибірка:", tuple(train_x.shape), " перевірочна:", tuple(test_x.shape))
print("класів           :", len(SHAPE_NAMES), "→", ", ".join(SHAPE_NAMES))
print("прикладів на клас:", int((train_y == 0).sum()))

## Замір 1 · Ціна вікон проти повної уваги

Повна увага рахує вагу для кожної пари токенів: `n × n` пар. Віконна ділить
сітку на вікна `w × w` і рахує увагу всередині кожного окремо.

Порахуємо обидві величини двома способами: прямим підрахунком (скільки вікон
× скільки пар у кожному) і за формулою `n × w²`. Якщо вони збігаються — формула
правильна.

In [ ]:
PATCH = 4
WINDOW = 7

print(f"{"зображення":>11} {"сітка":>8} {"токенів":>8} {"повна n²":>13} "
      f"{"вікон":>6} {"вікна: прямо":>13} {"вікна: n·w²":>12} {"дешевше":>8}")
print("-" * 88)

for image_side in (28, 56, 112, 224, 448, 896):
    grid = image_side // PATCH              # скільки токенів по стороні
    tokens = grid * grid
    full_pairs = tokens * tokens

    if grid % WINDOW != 0:
        continue                            # вікно має ділити сітку без остачі

    n_windows = (grid // WINDOW) ** 2
    # у кожному вікні w² токенів, отже пар у ньому (w²)² = w⁴
    direct = n_windows * (WINDOW * WINDOW) ** 2
    by_formula = tokens * WINDOW * WINDOW
    assert direct == by_formula, "формула n·w² розійшлася з прямим підрахунком!"

    print(f"{image_side:>4}×{image_side:<6} {grid:>3}×{grid:<4} {tokens:>8} {full_pairs:>13} "
          f"{n_windows:>6} {direct:>13} {by_formula:>12} {full_pairs // by_formula:>7}×")

print()
print("✅ формула n × w² скрізь збіглася з прямим підрахунком")

Тепер найголовніше — не самі числа, а те, **як вони ростуть**. Подвоїмо
сторону зображення й подивимось, у скільки разів подорожчає кожен варіант.

In [ ]:
print("подвоюємо сторону зображення:")
print(f"{"сітка":>9} {"токенів":>8} {"повна ×":>9} {"вікна ×":>9}")
print("-" * 40)

previous_full = None
previous_window = None
for grid in (7, 14, 28, 56, 112):
    tokens = grid * grid
    full_pairs = tokens * tokens
    window_pairs = tokens * WINDOW * WINDOW
    if previous_full is None:
        print(f"{grid:>3}×{grid:<5} {tokens:>8} {"—":>9} {"—":>9}")
    else:
        print(f"{grid:>3}×{grid:<5} {tokens:>8} "
              f"{full_pairs / previous_full:>8.0f}× {window_pairs / previous_window:>8.0f}×")
    previous_full, previous_window = full_pairs, window_pairs

print()
print("Токенів щоразу вчетверо більше. Повна увага дорожчає в 16 разів —")
print("це квадрат. Віконна — рівно вчетверо, тобто лінійно за кількістю токенів.")

## Замір 2 · Власна віконна увага

Тепер напишемо віконну увагу самі. Вона складається з трьох кроків:

1. **Розрізати** сітку токенів на вікна — з `(B, grid·grid, dim)` зробити
   `(B·вікон, w·w, dim)`.
2. Порахувати звичайну увагу окремо в кожному вікні.
3. **Зібрати** назад у ту саму форму, що була на вході.

Головна перевірка: якщо поставити вікно на всю сітку, віконна увага мусить
дати **точно те саме**, що звичайна повна.

In [ ]:
def window_partition(tokens, grid, window):
    '''Ріже сітку токенів на квадратні вікна.

    Вхід  (batch, grid·grid, dim) → вихід (batch·вікон, window·window, dim).
    '''
    batch, n_tokens, dim = tokens.shape
    per_side = grid // window                       # скільки вікон по стороні
    x = tokens.reshape(batch, per_side, window, per_side, window, dim)
    # переставляємо осі так, щоб обидва номери вікна опинились поруч із батчем
    x = x.permute(0, 1, 3, 2, 4, 5)
    return x.reshape(batch * per_side * per_side, window * window, dim)


def window_merge(windows, grid, window, batch):
    '''Зворотна операція: збирає вікна назад у сітку.'''
    per_side = grid // window
    x = windows.reshape(batch, per_side, per_side, window, window, -1)
    x = x.permute(0, 1, 3, 2, 4, 5)
    return x.reshape(batch, grid * grid, -1)


def window_attention(attention, tokens, grid, window, shift=0, mask=None):
    '''Увага всередині вікон, з необовʼязковим циклічним зсувом і маскою.'''
    batch, n_tokens, dim = tokens.shape
    x = tokens.reshape(batch, grid, grid, dim)
    if shift:
        # прокручуємо саму сітку, а не межі вікон — так вікна лишаються рівними
        x = torch.roll(x, shifts=(-shift, -shift), dims=(1, 2))

    pieces = window_partition(x.reshape(batch, n_tokens, dim), grid, window)
    # маску будували на одне зображення — повторюємо її на весь батч
    full_mask = None if mask is None else mask.repeat(batch, 1, 1)
    attended, _ = attention(pieces, pieces, pieces,
                            attn_mask=full_mask, need_weights=False)

    y = window_merge(attended, grid, window, batch).reshape(batch, grid, grid, dim)
    if shift:
        y = torch.roll(y, shifts=(shift, shift), dims=(1, 2))
    return y.reshape(batch, n_tokens, dim)


DIM = 16
HEADS = 2
torch.manual_seed(7)
attention_layer = nn.MultiheadAttention(DIM, HEADS, batch_first=True)
attention_layer.eval()

# сітка 7×7 і вікно 7×7 — це рівно одне вікно, тобто повна увага
torch.manual_seed(1)
probe = torch.randn(3, 49, DIM)
with torch.no_grad():
    ours = window_attention(attention_layer, probe, grid=7, window=7)
    library, _ = attention_layer(probe, probe, probe, need_weights=False)

assert np.allclose(ours.numpy(), library.numpy(), atol=1e-6), "віконна увага розійшлася!"
print("вікно = вся сітка == повна увага :", torch.equal(ours, library))
print("макс |різниця|                   :", float((ours - library).abs().max()))
print()
print("✅ наша віконна увага при вікні на всю сітку — це та сама повна увага")

## Замір 3 · Вікна не спілкуються

Тепер доказ, заради якого й вигадали зсув. Він не потребує навчання зовсім.

Беремо сітку 14×14 (це 196 токенів) і вікно 7×7 — виходить чотири вікна.
Беремо випадковий набір токенів, робимо його копію й **міняємо в копії рівно
один токен**. Проганяємо обидва набори крізь віконну увагу з однаковими вагами
й рахуємо, у скількох токенах виходи розійшлися.

Якщо вікна справді ізольовані, зміна не вийде за межі свого вікна — тобто
зачепить рівно 49 токенів, скільки шарів не став.

In [ ]:
GRID = 14
WINDOW = 7
SHIFT = WINDOW // 2          # зсув на пів вікна: 7 // 2 = 3

CHANGED_ROW, CHANGED_COL = 3, 6      # край лівого верхнього вікна


def make_pair():
    '''Два однакові набори токенів, що відрізняються рівно одним токеном.'''
    torch.manual_seed(2)
    original = torch.randn(1, GRID * GRID, DIM)
    changed = original.clone()
    changed[0, CHANGED_ROW * GRID + CHANGED_COL] += 3.0
    return original, changed


def count_touched(layer_shifts, mask=None):
    '''Скільки токенів на виході відрізняються після кожного шару.'''
    original, changed = make_pair()
    counts = []
    with torch.no_grad():
        for use_shift in layer_shifts:
            shift = SHIFT if use_shift else 0
            layer_mask = mask if use_shift else None
            original = window_attention(attention_layer, original, GRID, WINDOW, shift, layer_mask)
            changed = window_attention(attention_layer, changed, GRID, WINDOW, shift, layer_mask)
            difference = (original - changed).abs().reshape(GRID, GRID, DIM).amax(-1)
            counts.append(int((difference > 1e-6).sum()))
    return counts


plain = count_touched([0, 0, 0, 0])
print("змінили 1 токен із", GRID * GRID)
print("звичайні вікна, зачеплено токенів після шарів 1-4:", plain)
print()
print("Це рівно розмір одного вікна:", WINDOW * WINDOW)
print("Скільки шарів не додавай — інформація не виходить за межу вікна.")

## Замір 4 · Зсунуті вікна з маскою

Лікування: у кожному другому шарі прокручуємо сітку на пів вікна (`torch.roll`),
а потім маскою забороняємо пари, які зійшлися лише через прокручування.

Маска будується так: позначаємо, які рядки й стовпці прийшли з протилежного
краю, і забороняємо пари з різними позначками.

In [ ]:
def build_shift_mask(grid, window, shift, heads):
    '''True означає «цій парі дивитись одне на одного заборонено».

    Прокручування підклеює верхній край до нижнього. Такі токени опиняються
    в одному вікні, хоча в зображенні вони на протилежних кінцях.
    '''
    # для кожного рядка прокрученої сітки: чи прийшов він з протилежного краю
    wrapped = torch.tensor([(i + shift) >= grid for i in range(grid)], dtype=torch.long)
    # область токена = пара позначок (рядок, стовпець) → чотири можливі значення
    region = (wrapped.reshape(grid, 1) * 2 + wrapped.reshape(1, grid))
    region = region.reshape(1, grid * grid, 1).float()

    in_windows = window_partition(region, grid, window).squeeze(-1)   # (вікон, w·w)
    blocked = in_windows.unsqueeze(1) != in_windows.unsqueeze(2)
    # MultiheadAttention чекає окрему маску на кожну голову
    return blocked.repeat_interleave(heads, 0)


shift_mask = build_shift_mask(GRID, WINDOW, SHIFT, HEADS)
per_window = shift_mask[::HEADS]

print("скільки пар маска забороняє в кожному з чотирьох вікон:")
for k in range(per_window.shape[0]):
    blocked = int(per_window[k].sum())
    total = per_window[k].numel()
    print(f"   вікно {k}: {blocked:>5} із {total} ({100 * blocked / total:.0f} %)")
print()
print("усього заблоковано:", int(per_window.sum()), "із", per_window.numel())

Тепер той самий підрахунок, але з чергуванням: непарний шар — звичайні
вікна, парний — зсунуті. І третій варіант для чесності: **усі** шари зсунуті
однаково.

In [ ]:
alternating = count_touched([0, 1, 0, 1], mask=shift_mask)
all_shifted = count_touched([1, 1, 1, 1], mask=shift_mask)

print(f"{"режим":<28} {"1":>5} {"2":>5} {"3":>5} {"4":>5}")
print("-" * 54)
print(f"{"звичайні вікна":<28}", " ".join(f"{v:>5}" for v in plain))
print(f"{"чергування зі зсунутими":<28}", " ".join(f"{v:>5}" for v in alternating))
print(f"{"усі шари зсунуті однаково":<28}", " ".join(f"{v:>5}" for v in all_shifted))
print()
print("Після другого шару зачеплено", alternating[1], "токенів замість", plain[1], "—")
print("інформація перетекла через межу вікна.")
print("Після третього —", alternating[2], "з", GRID * GRID, ", тобто вся сітка.")
print()
print("Головне: «усі шари зсунуті» дає той самий результат, що «жоден не зсунутий».")
print("Працює не зсув сам по собі, а РІЗНИЦЯ між сусідніми шарами.")

## Замір 5 · Ієрархія: обʼєднання патчів

Друга ідея Swin — піраміда. Обʼєднання патчів бере квадрат 2×2 сусідніх токенів,
зчіплює їхні вектори в один довжини `4C` і пропускає крізь `LayerNorm(4C)`
і `Linear(4C → 2C)` без зсуву.

Роздільність падає вдвічі, каналів стає вдвічі більше — це правило подвоєння
з теми 09.

In [ ]:
def swin_stages(image_side=224, patch=4, channels=96, multiplier=2, window=7):
    '''Форми на кожній із чотирьох стадій.'''
    rows = []
    grid = image_side // patch
    for stage in range(4):
        tokens = grid * grid
        window_pairs = tokens * window * window if grid % window == 0 else None
        rows.append({
            "стадія": stage + 1,
            "сітка": f"{grid}×{grid}",
            "каналів": channels,
            "токенів": tokens,
            "чисел у карті": tokens * channels,
            "пар: вікна": window_pairs,
            "пар: повна": tokens * tokens,
        })
        grid //= 2
        channels *= multiplier
    return rows


print(f"{"стадія":>7} {"сітка":>8} {"каналів":>8} {"токенів":>8} "
      f"{"чисел у карті":>14} {"пар: вікна":>11} {"пар: повна":>12}")
print("-" * 76)
for row in swin_stages():
    print(f"{row["стадія"]:>7} {row["сітка"]:>8} {row["каналів"]:>8} {row["токенів"]:>8} "
          f"{row["чисел у карті"]:>14} {row["пар: вікна"]:>11} {row["пар: повна"]:>12}")

print()
print("ResNet на тому самому вході 224×224 — для порівняння:")
for resolution, ch in ((56, 64), (28, 128), (14, 256), (7, 512)):
    print(f"   {resolution:>2}×{resolution:<3} каналів {ch:>4}")
print()
print("Роздільності збігаються повністю — саме тому Swin можна поставити")
print("в готовий детектор замість ResNet, нічого більше не переробляючи.")

In [ ]:
def patch_merging_weights(channels_in, multiplier=2):
    '''Ваги одного шару обʼєднання: LayerNorm(4C) плюс Linear(4C → kC) без зсуву.'''
    norm = 2 * 4 * channels_in                                # ваги й зсув LayerNorm
    linear = 4 * channels_in * (multiplier * channels_in)     # bias=False
    return norm + linear


total = 0
print("три шари обʼєднання в swin_t:")
for channels_in in (96, 192, 384):
    weights = patch_merging_weights(channels_in)
    total += weights
    print(f"   {channels_in:>3} → {2 * channels_in:>3} каналів: "
          f"{4 * channels_in} → {2 * channels_in}, ваг {weights:>9,}".replace(",", " "))
print(f"   разом: {total:,}".replace(",", " "),
      f"— це {100 * total / 28288354:.1f} % від усієї мережі")

## Замір 6 · Розбір справжнього `swin_t`

`weights=None` означає, що нічого не завантажується з мережі — це чиста
арифметика будови.

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())


swin = models.swin_t(weights=None)
features = swin.features

parts = [
    ("проєкція патча", features[0]),
    ("стадія 1 · 2 блоки", features[1]),
    ("обʼєднання 1→2", features[2]),
    ("стадія 2 · 2 блоки", features[3]),
    ("обʼєднання 2→3", features[4]),
    ("стадія 3 · 6 блоків", features[5]),
    ("обʼєднання 3→4", features[6]),
    ("стадія 4 · 2 блоки", features[7]),
    ("фінальна норма", swin.norm),
    ("класифікатор", swin.head),
]

whole = count(swin)
print(f"{"частина":<24}{"ваг":>13}{"частка":>9}")
print("-" * 46)
checksum = 0
for name, module in parts:
    weights = count(module)
    checksum += weights
    print(f"{name:<24}{weights:>13,}{100 * weights / whole:>8.1f} %".replace(",", " "))
print("-" * 46)
print(f"{"усього":<24}{whole:>13,}".replace(",", " "))
assert checksum == whole, "розклад не зійшовся з повною сумою!"
print("✅ сума частин збіглася з повною кількістю ваг")

In [ ]:
# тепер той самий розклад, але за призначенням, а не за стадіями
attention_weights = 0
mlp_weights = 0
norm_weights = 0
bias_table_weights = 0

for stage_index in (1, 3, 5, 7):
    for block in features[stage_index]:
        for name, parameter in block.named_parameters():
            if name.startswith("attn"):
                attention_weights += parameter.numel()
                if "relative_position_bias_table" in name:
                    bias_table_weights += parameter.numel()
            elif name.startswith("mlp"):
                mlp_weights += parameter.numel()
            elif name.startswith("norm"):
                norm_weights += parameter.numel()

print(f"{"віконна увага":<30}{attention_weights:>12,}{100 * attention_weights / whole:>8.1f} %".replace(",", " "))
print(f"{"  · з них відносне зміщення":<30}{bias_table_weights:>12,}{100 * bias_table_weights / whole:>8.2f} %".replace(",", " "))
print(f"{"MLP усередині блоків":<30}{mlp_weights:>12,}{100 * mlp_weights / whole:>8.1f} %".replace(",", " "))
print(f"{"LayerNorm усередині блоків":<30}{norm_weights:>12,}{100 * norm_weights / whole:>8.2f} %".replace(",", " "))
print()
print("Увага — НЕ найдорожча частина трансформера:")
print(f"на MLP припадає у {mlp_weights / attention_weights:.1f} раза більше ваг, ніж на увагу.")

In [ ]:
# відносне позиційне зміщення: одне число на кожне взаємне положення й голову
print("таблиця відносних зміщень для вікна 7×7:")
print("різниць по рядку:", 2 * 7 - 1, " по стовпцю:", 2 * 7 - 1,
      " разом положень:", (2 * 7 - 1) ** 2)
print()
for dim, heads in ((96, 3), (192, 6), (384, 12), (768, 24)):
    print(f"   dim {dim:>3}, голів {heads:>2}: {(2 * 7 - 1) ** 2} × {heads} = {169 * heads:>5} ваг на блок")
print()
print("усього по мережі:", bias_table_weights, "ваг —",
      f"{100 * bias_table_weights / whole:.2f} % від swin_t")
print("абсолютне кодування лише для сітки 56×56 при 96 каналах коштувало б:",
      56 * 56 * 96, "ваг")

In [ ]:
# один блок стадії 3 — щоб було видно кожне число
block = features[5][0]
print("блок стадії 3, dim = 384:")
for name, module in block.named_children():
    weights = count(module)
    if weights:
        print(f"   {name:<10} {type(module).__name__:<24}{weights:>12,}".replace(",", " "))
print()
print("усередині віконної уваги:")
for name, parameter in block.attn.named_parameters():
    print(f"   {name:<32}{str(tuple(parameter.shape)):>14}{parameter.numel():>10,}".replace(",", " "))
print()
print(f"весь блок: {count(block):,}".replace(",", " "))

## Замір 7 · `convnext_tiny` — що в ній від згортки, а що від трансформера

ConvNeXt — згорткова мережа, яку переодягли в трансформер: великі ядра 7×7,
GELU замість ReLU, LayerNorm замість BatchNorm, менше нормалізацій.
Подивимось, скільки в ній чого.

In [ ]:
convnext = models.convnext_tiny(weights=None)
vit = models.vit_b_16(weights=None)

print("три моделі одного покоління:")
for name, model in (("vit_b_16", vit), ("swin_t", swin), ("convnext_tiny", convnext)):
    print(f"   {name:<16}{count(model):>12,}".replace(",", " "))
print()
print("різниця між swin_t і convnext_tiny:",
      f"{abs(count(convnext) - count(swin)):,}".replace(",", " "),
      f"— це {100 * abs(count(convnext) - count(swin)) / count(swin):.1f} % ")
print()
print("стем обох мереж:")
print("   swin_t        :", count(swin.features[0]), "ваг")
print("   convnext_tiny :", count(convnext.features[0]), "ваг")
assert count(swin.features[0]) == count(convnext.features[0])
print("✅ збігається до одиниці — це той самий шар під двома іменами")

In [ ]:
print("один блок ConvNeXt (стадія 1):")
print(convnext.features[1][0].block)

In [ ]:
# розкладаємо ваги ConvNeXt за типом шару
by_kind = {}
depthwise_weights = 0
depthwise_count = 0

for name, module in convnext.named_modules():
    kind = type(module).__name__
    if kind in ("Conv2d", "Linear", "LayerNorm", "LayerNorm2d"):
        weights = count(module)
        got = by_kind.get(kind, [0, 0])
        by_kind[kind] = [got[0] + 1, got[1] + weights]
    if isinstance(module, nn.Conv2d) and module.groups == module.in_channels and module.groups > 1:
        depthwise_weights += count(module)     # глибинна згортка — єдине, що дивиться на сусідів
        depthwise_count += 1

whole_next = count(convnext)
print(f"{"тип шару":<16}{"штук":>6}{"ваг":>13}{"частка":>9}")
print("-" * 44)
for kind, (how_many, weights) in sorted(by_kind.items(), key=lambda kv: -kv[1][1]):
    print(f"{kind:<16}{how_many:>6}{weights:>13,}{100 * weights / whole_next:>8.1f} %".replace(",", " "))
print("-" * 44)
print(f"{"глибинні 7×7":<16}{depthwise_count:>6}{depthwise_weights:>13,}"
      f"{100 * depthwise_weights / whole_next:>8.1f} %".replace(",", " "))
print()
print("Усе просторове змішування — тобто всі шари, що взагалі дивляться на")
print(f"сусідів — важить {depthwise_weights:,}".replace(",", " "),
      f"ваг, або {100 * depthwise_weights / whole_next:.1f} % мережі.")

In [ ]:
resnet = models.resnet50(weights=None)

conv_weights = sum(count(m) for _, m in resnet.named_modules() if isinstance(m, nn.Conv2d))
print("ResNet-50 для порівняння:")
print(f"   усього ваг       : {count(resnet):,}".replace(",", " "))
print(f"   у згортках       : {conv_weights:,}".replace(",", " "),
      f"— {100 * conv_weights / count(resnet):.1f} %")
print()
print("Два зовсім різні способи витратити однаковий бюджет параметрів:")
print("у ResNet-50 згортки тримають 92 % ваг, у ConvNeXt просторове")
print("змішування — близько одного відсотка.")

In [ ]:
# скільки нормалізацій і активацій в одному блоці
def count_kinds(module, kinds):
    return sum(1 for _, m in module.named_modules() if type(m).__name__ in kinds)


convnext_block = convnext.features[1][0]
resnet_block = resnet.layer1[0]

print("нормалізацій + активацій в одному блоці:")
print("   ConvNeXt          :", count_kinds(convnext_block, ("LayerNorm", "GELU")))
print("   ResNet-50         :", count_kinds(resnet_block, ("BatchNorm2d", "ReLU")))
print()
print("скільки блоків у кожній стадії:")
print("   ConvNeXt-T:", [len(convnext.features[i]) for i in (1, 3, 5, 7)])
print("   Swin-T    :", [len(swin.features[i]) for i in (1, 3, 5, 7)])
print("   ResNet-50 :", [len(getattr(resnet, f"layer{i}")) for i in (1, 2, 3, 4)])
print()
print("Співвідношення 1:1:3:1 ConvNeXt узяла в Swin.")

## Замір 8 · Три моделі однакового розміру

Останній замір, і єдиний, що потребує навчання. Три моделі:

1. **повна увага** — патч 2×2 дає сітку 14×14, тобто 196 токенів, увага
   на всіх одразу;
2. **вікна 7×7 зі зсувом** — те саме, але увага всередині вікон, у другому
   блоці зі зсувом;
3. **згортка** — маленька CNN, підігнана під той самий розмір.

Перші дві моделі відрізняються **рівно одним**: розміром вікна. Тому ваг
у них однаково до одиниці, і будь-яка різниця між ними — від уваги.

Три зерна на кожну точку: різниця, менша за розкид від зерна, різницею не є.

⚠️ **Точності в тебе збіжаться з нашими, а секунди — ні.** Точність задана
зернами й від завантаження машини не залежить зовсім. Час залежить, і сильно:
у двох наших прогонах виграш віконної моделі над повною увагою вийшов
1,8 раза і 1,2 раза. Тому нижче ми друкуємо ще й величину, яка не пливе, —
кількість елементів матриці уваги.

> ⏱ Ця клітинка навчає девʼять мереж і йде близько трьох з половиною хвилин.

In [ ]:
class WindowBlock(nn.Module):
    '''Блок трансформера, у якому увага рахується всередині вікон.'''

    def __init__(self, dim, heads, feedforward, grid, window, shift=0):
        super().__init__()
        self.grid, self.window, self.shift = grid, window, shift
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(dim, feedforward), nn.GELU(),
                                 nn.Linear(feedforward, dim))

    def forward(self, x):
        # норма спереду, як у темі 20: магістраль залишкового потоку лишається чистою
        attended = window_attention(self.attn, self.norm1(x), self.grid,
                                    self.window, self.shift)
        x = x + attended
        return x + self.mlp(self.norm2(x))


class TinyWindowViT(nn.Module):
    '''Крихітний трансформер: вікно = всій сітці означає звичайний ViT.'''

    def __init__(self, patch=2, dim=32, depth=2, heads=4, feedforward=64,
                 window=None, shift=False, n_classes=6):
        super().__init__()
        grid = 28 // patch
        window = window or grid                    # замовчування — повна увага
        self.to_patch = nn.Conv2d(1, dim, kernel_size=patch, stride=patch)
        self.pos = nn.Parameter(torch.zeros(1, grid * grid, dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([
            WindowBlock(dim, heads, feedforward, grid, window,
                        window // 2 if (shift and i % 2 == 1) else 0)
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        tokens = self.to_patch(x).flatten(2).transpose(1, 2) + self.pos
        for block in self.blocks:
            tokens = block(tokens)
        return self.head(self.norm(tokens).mean(1))


class TinyCNN(nn.Module):
    '''Три згортки 3×3 → глобальне усереднення → лінійний шар.'''

    def __init__(self, n_classes=6, c1=16, c2=32, c3=56):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, c1, 3, padding=1, bias=False), nn.BatchNorm2d(c1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(c1, c2, 3, padding=1, bias=False), nn.BatchNorm2d(c2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(c2, c3, 3, padding=1, bias=False), nn.BatchNorm2d(c3), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(c3, n_classes)

    def forward(self, x):
        return self.head(self.body(x).flatten(1))


print("ваг у моделях:")
print("   повна увага       :", count(TinyWindowViT()))
print("   вікна 7×7 зі зсувом:", count(TinyWindowViT(window=7, shift=True)))
print("   згортка           :", count(TinyCNN()))

In [ ]:
def train_and_measure(model, epochs=14, lr=3e-3, batch_size=64, seed=0):
    '''Навчає модель і повертає точність на перевірці та витрачений час.'''
    torch.manual_seed(seed)
    # AdamW, а не SGD: на SGD трансформери вчаться погано, і легко видати
    # властивість налаштувань за властивість архітектури
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    n = train_x.shape[0]
    started = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(n)
        for start in range(0, n, batch_size):
            batch = order[start:start + batch_size]
            loss = F.cross_entropy(model(train_x[batch]), train_y[batch])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    elapsed = time.perf_counter() - started

    model.eval()
    with torch.no_grad():
        accuracy = (model(test_x).argmax(1) == test_y).float().mean().item()
    return accuracy, elapsed


experiments = [
    ("повна увага", lambda: TinyWindowViT()),
    ("вікна 7×7", lambda: TinyWindowViT(window=7, shift=True)),
    ("згортка", lambda: TinyCNN()),
]

results = {}
for label, build in experiments:
    accuracies = []
    times = []
    for seed in (0, 1, 2):
        torch.manual_seed(seed)
        accuracy, elapsed = train_and_measure(build(), seed=seed)
        accuracies.append(accuracy)
        times.append(elapsed)
    results[label] = (accuracies, times, count(build()))
    spread = (max(accuracies) - min(accuracies)) / 2
    print(f"{label:<12} ваг {count(build()):>6}  "
          f"точність {np.mean(accuracies):.3f} ±{spread:.3f}  "
          f"прогони {[round(a, 3) for a in accuracies]}  "
          f"час {np.mean(times):.1f} с")

In [ ]:
full_accuracy = np.mean(results["повна увага"][0])
window_accuracy = np.mean(results["вікна 7×7"][0])
conv_accuracy = np.mean(results["згортка"][0])

full_spread = (max(results["повна увага"][0]) - min(results["повна увага"][0])) / 2
window_spread = (max(results["вікна 7×7"][0]) - min(results["вікна 7×7"][0])) / 2
biggest_spread = max(full_spread, window_spread)

print("Висновки, зроблені за правилом теми 18: різниця, менша за розкид,")
print("різницею не є.")
print()
difference = abs(full_accuracy - window_accuracy)
print(f"повна увага проти вікон: {difference:.3f} при розкиді ±{biggest_spread:.3f}")
if difference > biggest_spread:
    better = "вікна" if window_accuracy > full_accuracy else "повна увага"
    print(f"   → різниця більша за розкид, отже {better} справді кращі")
else:
    print("   → різниця менша за розкид, отже різниці ми НЕ виявили")
print()
print(f"згортка проти вікон: {conv_accuracy - window_accuracy:.3f}")
print(f"   → відрив великий, і це той самий висновок, що в темі 20")
print()
window_time = np.mean(results["вікна 7×7"][1])
full_time = np.mean(results["повна увага"][1])
print(f"час: вікна {window_time:.1f} с проти повної уваги {full_time:.1f} с "
      f"при однакових вагах — у {full_time / window_time:.1f} раза швидше")
print("   ⚠️ секунди залежать від завантаження машини; у тебе вони будуть інші")
print()

# величина, яка не пливе від прогону до прогону
full_matrix = 196 * 196
window_matrix = 4 * 49 * 49
print("а ось число, яке не залежить ані від машини, ані від прогону:")
print(f"   повна увага: одна матриця 196×196 = {full_matrix} елементів")
print(f"   вікна 7×7  : чотири матриці 49×49 = {window_matrix} елементів")
print(f"   рівно в {full_matrix // window_matrix} рази менше — це і є ефект вікон")
print()
print(f"⏱ увесь зошит: {time.perf_counter() - notebook_started:.0f} с")

## Завдання

### 🟢 Рівень 1

Візьми функцію `count_touched` і перевір, що станеться, якщо чергування
починається не зі звичайного шару, а зі зсунутого: `[1, 0, 1, 0]`.
Чи буде результат такий самий, як у `[0, 1, 0, 1]`?

### 🟡 Рівень 2

Додай до `swin_stages` ще одну колонку: у скільки разів віконна увага дешевша
за повну на кожній стадії. Поясни словами, чому на четвертій стадії число
дорівнює одиниці.

### 🔴 Рівень 3

Побудуй маску для вікна 4 на сітці 8×8 (зсув 2) і доведи числом, що
з чергуванням інформація дістає всієї сітки за три шари, а без чергування —
не дістає ніколи.

Повні умови — у [домашньому завданні](homework.html).